# [INFO-H515 - Big Data Scalable Analytics](https://uv.ulb.ac.be/course/view.php?id=85246?username=guest)

## TP 5 - Examn question : Recommendation System

#### *Gianluca Bontempi, Cédric Simar, and Martin Colot - materials from Jacopo De Stefani and Theo Verhelst*

####  2025


## General imports

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from pyspark.sql import SparkSession

## Data generation

In [188]:
def genData(N, n, NA, random_seed):
    np.random.seed(random_seed)

    #Inputs and the weights of the linear combination are drawn at random
    X = np.random.rand(N, n)

    for i in range(NA):
      a = np.random.randint(0, len(X))
      b = np.random.randint(0, len(X[0]))
      X[a, b] = np.nan

    print("Number of Users :", N)
    print("Number of products :", n)

    print("Dimension of X :", X.shape)

    return X

In [189]:
X = genData(100, 300, NA=20000, random_seed=10)

X[0, 0] = np.nan

Number of Users : 100
Number of products : 300
Dimension of X : (100, 300)


In [190]:
data = X
data = np.concatenate((np.arange(len(data)).reshape(1, -1).transpose(), data), axis=1)

In [191]:
data.shape

(100, 301)

In [192]:
pd.DataFrame(data).head()

,0,1,2,3,4,5,6,7,8,9,...,291,292,293,294,295,296,297,298,299,300
0,0.0,NaN,NaN,NaN,NaN,0.498507,NaN,0.198063,0.760531,0.169111,...,0.086969,0.783964,NaN,0.584056,NaN,0.880792,0.600954,NaN,NaN,0.297484
1,1.0,0.691775,0.189029,NaN,NaN,0.757286,NaN,0.082620,0.482072,NaN,...,0.226360,0.004408,0.139780,0.295764,0.195594,NaN,0.209992,0.778492,NaN,0.639340
2,2.0,0.167696,0.924683,NaN,0.154814,0.723634,NaN,0.829665,NaN,0.961301,...,0.398776,0.387142,0.187320,0.423385,0.737978,0.990120,0.068836,NaN,NaN,NaN
3,3.0,0.297540,NaN,NaN,NaN,NaN,NaN,0.504510,0.177767,0.231424,...,0.124048,NaN,0.230772,NaN,NaN,NaN,NaN,0.962375,0.731044,NaN
4,4.0,NaN,NaN,0.967873,NaN,0.305025,0.590144,0.613003,0.207448,NaN,...,0.487798,NaN,NaN,NaN,NaN,0.624131,0.301486,NaN,0.493038,0.645543


## Spark Session

In [193]:
# Initialize Spark Session (local mode)
# Start Spark session with local master and 2 cores
spark = SparkSession \
    .builder \
    .master("local[2]") \
    .appName("LinearRegression") \
    .getOrCreate()

# Let us retrieve the sparkContext object
sc = spark.sparkContext

spark.conf.set("spark.sql.shuffle.partitions", "2") # reduce number of partitions to run on google Colab

In [194]:
#Let us use 2 partitions
B = 2
ratingMatrix = sc.parallelize(data, B).cache()

In [195]:
np.array(ratingMatrix.collect()).shape

(100, 301)

**Exercise 1:**


The task consists of searching for the nearest-neighbour user in the rating matrix.
In this matrix, each line corresponds to a user, and each column to a product. The first column gives the ID of the user and the other the ratings of the products by the users. If the user has given no rating for a product, the matrix contians a $NaN$.


\begin{equation}
\begin{pmatrix}
  0       & X_{1,1}   & X_{1,2}   & \cdots  & X_{1,n}  \\
  1       & X_{2,1}   & X_{2,2}   & \cdots  & X_{2,n} \\
  \cdots \\
  N       & X_{N,1}   & X_{N,2}   & \cdots  & X_{N,n} \\
\end{pmatrix}
\end{equation}



Given a rating matrix and a target user and product (identified by their ids) without known rating, write a map-reduce code (RDD-based) that estimates the missing value by finding the rating of the closest user for this product.
The rating distance between two users is defined as the average difference between the ratings of products rated by the two users.



**Your Solution**

**Solution**

In [196]:
targetUserId = 0
targetProductId = 0

targetUser = ratingMatrix.filter(lambda x: x[0] == targetUserId).collect()[0]
targetUser_bc = sc.broadcast(targetUser) # broadcast ratings of target user to all workers

# 1. remove the user who did not give a rating for the target product
a = ratingMatrix.filter(lambda x: not np.isnan(x[targetProductId+1]))

# 2. distribute the common ratings (only usefull if the number of users and products are both Big and you have more workers than users)
b = a.flatMap(lambda x : [(x[0], x[i], targetUser_bc.value[i]) for i in range(1, len(x)) if (not np.isnan(x[i]) and not np.isnan(targetUser_bc.value[i]))])

# 3. compute the rating differences. "1" will be used to find the denominator of the average
c = b.map(lambda x : (x[0], (abs(x[1] - x[2]), 1)))

# 4. find the sum of ratings differences and number of common ratings
d = c.reduceByKey(lambda x, y : (x[0] + y[0], x[1] + y[1]))

# 5. compute average
e = d.map(lambda x : (x[0], x[1][0] / x[1][1]))

# 6. find user with smallest average
closestUser = e.reduce(lambda x, y: x if x[1] < y[1] else y)[0]

# 7. find rating of target product by the other user
ratingEstimate = ratingMatrix.filter(lambda x: x[0] == closestUser).collect()[0][targetProductId+1]

In [197]:
closestUser, ratingEstimate

(np.float64(75.0), np.float64(0.558520139096307))

**Exercise 2:**


The question is the same as the Exercice 1, but you must estimate the target missing rating by finding the closest product (instead of the closest user).
You then estimate the target rating as the rating that the target user has given to the closest product.
However, as the rating matrix is very big, you cannot transpose it.



**Your Solution**

**Solution**

In [198]:
targetUserId = 0
targetProductId = 0

targetUser = ratingMatrix.filter(lambda x: x[0] == targetUserId).collect()[0]
targetUser_bc = sc.broadcast(targetUser) # broadcast ratings of target user to all workers

# 1. compute the rating difference for each user who have rated the target product, with each of their rated products, that cas also rated by the target user, and distribute the samples
# "1" is added to compute the denominator of the average
a = ratingMatrix.flatMap(lambda x: [(j, (np.abs(x[j] - x[targetProductId+1]), 1)) for j in range(1, len(x)) if j != targetProductId+1
                                                                                                          and not np.isnan(targetUser_bc.value[j])
                                                                                                          and not np.isnan(x[j])
                                                                                                          and not np.isnan(x[targetProductId+1])])
# 2. compute sum and count of rating differences for each product with the targte product
b = a.reduceByKey(lambda x, y: (x[0] + y[0], x[1] + y[1]))

# 3. compute averages
c = b.map(lambda x: (x[0], x[1][0] / x[1][1]))

# 4. find product, with the closest rating to the target product
closestProduct = c.reduce(lambda x, y: x if x[1] < y[1] else y)[0]

# 5. estimate the rating of the target product as the rating of the closest product given by yhe targte user
ratingEstimate = targetUser[int(closestProduct)]

In [199]:
closestProduct, ratingEstimate

(167, np.float64(0.3851135995654865))